# Signal Research Framework

**Learning objectives:**
- Understand different types of trading signals
- Use the signal registry to test factors
- Blend multiple signals together
- Test signals with proper walk-forward validation

This notebook demonstrates the signal research framework.

## 1. Setup

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

project_root = Path('/Users/zelin/Desktop/PA Investment/Invest_strategy')
sys.path.insert(0, str(project_root))

from research.signals import (
    list_signals, get_signal, SignalBlender,
    MomentumSignal, MeanReversionSignal, RSISignal, MACDSignal,
    BollingerPositionSignal, SMACrossoverSignal, VolatilitySignal
)

print("Setup complete")
print(f"Available signals: {list_signals()}")

## 2. Load Data

In [ ]:
# Load multiple assets for multi-asset research
assets = {
    'SPY': 'SPY',    # S&P 500
    'TLT': 'TLT',    # 20+ Year Treasury
    'GLD': 'GLD',    # Gold
    'UUP': 'UUP',    # Dollar ETF
}

start_date = '2015-01-01'
end_date = '2024-12-31'

# Load single asset for demonstration
ticker = 'SPY'
df = yf.download(ticker, start=start_date, end=end_date, progress=False)
df = df.droplevel('Ticker', axis=1)
df = df.reset_index()
df.columns = ['date', 'open', 'high', 'low', 'close', 'volume']

print(f"Loaded {len(df)} days of {ticker} data")
print(df.tail())

## 3. Compute Individual Signals

In [ ]:
# Create price DataFrame
prices = pd.DataFrame({'close': df.set_index('date')['close']})

# Compute momentum signal
momentum = MomentumSignal(lookback=60, skip=21)
mom_signal = momentum.compute(prices)

# Compute mean reversion signal
mean_rev = MeanReversionSignal(lookback=20)
mr_signal = mean_rev.compute(prices)

# Compute RSI
rsi = RSISignal(lookback=14)
rsi_signal = rsi.compute(prices)

# Compute MACD
macd = MACDSignal()
macd_signal = macd.compute(prices)

# Compute Bollinger Band position
bb = BollingerPositionSignal(lookback=20, num_std=2)
bb_signal = bb.compute(prices)

# Plot signals
import matplotlib.pyplot as plt

fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True)

axes[0].plot(mom_signal, label='Momentum (60,21)', color='blue')
axes[0].legend()
axes[0].set_title('Momentum Signal')

axes[1].plot(mr_signal, label='Mean Reversion (20)', color='green')
axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[1].legend()
axes[1].set_title('Mean Reversion Signal')

axes[2].plot(rsi_signal, label='RSI (14)', color='purple')
axes[2].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[2].legend()
axes[2].set_title('RSI Signal (0 = oversold, +50 = overbought)')

axes[3].plot(macd_signal, label='MACD', color='orange')
axes[3].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[3].legend()
axes[3].set_title('MACD Histogram')

axes[4].plot(bb_signal, label='Bollinger Position', color='red')
axes[4].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[4].legend()
axes[4].set_title('Bollinger Band Position')

plt.tight_layout()
plt.show()

## 4. Convert Signals to Positions

In [ ]:
# Convert signals to positions
# Each signal has a to_positions() method

# Momentum: long when positive, short when negative
mom_positions = momentum.to_positions(mom_signal)

# Mean reversion: opposite - buy when negative (price below MA)
mr_positions = mean_rev.to_positions(mr_signal)

# RSI: oversold = buy, overbought = sell
rsi_positions = rsi.to_positions(rsi_signal)

# Plot position series
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)

axes[0].plot(mom_positions, label='Momentum Positions')
axes[0].legend()
axes[0].set_title('Momentum Positions (1=long, -1=short)')

axes[1].plot(mr_positions, label='Mean Reversion Positions')
axes[1].legend()
axes[1].set_title('Mean Reversion Positions')

axes[2].plot(rsi_positions, label='RSI Positions')
axes[2].legend()
axes[2].set_title('RSI Positions')

plt.tight_layout()
plt.show()

## 5. Signal Returns Analysis

In [ ]:
# Calculate returns for each signal strategy
returns = prices['close'].pct_change()

# Signal returns: position from yesterday * return today
mom_returns = mom_positions.shift(1) * returns
mr_returns = mr_positions.shift(1) * returns
rsi_returns = rsi_positions.shift(1) * returns

# Calculate cumulative returns
mom_cum = (1 + mom_returns.dropna()).cumprod()
mr_cum = (1 + mr_returns.dropna()).cumprod()
rsi_cum = (1 + rsi_returns.dropna()).cumprod()
buy_hold = (1 + returns.dropna()).cumprod()

# Plot
plt.figure(figsize=(12, 6))
plt.plot(mom_cum, label='Momentum')
plt.plot(mr_cum, label='Mean Reversion')
plt.plot(rsi_cum, label='RSI')
plt.plot(buy_hold, label='Buy & Hold', alpha=0.5)
plt.legend()
plt.title('Cumulative Returns by Signal Strategy')
plt.ylabel('Cumulative Return')
plt.grid(True, alpha=0.3)
plt.show()

# Calculate metrics
def calc_metrics(returns_series, name):
    total_ret = (1 + returns_series).prod() - 1
    sharpe = returns_series.mean() / returns_series.std() * np.sqrt(252)
    max_dd = ((1 + returns_series).cumprod().cummax() - (1 + returns_series).cumprod()).max()
    win_rate = (returns_series > 0).mean()
    print(f"{name:20s}: Return={total_ret*100:7.2f}%, Sharpe={sharpe:6.2f}, MaxDD={max_dd*100:6.2f}%, WinRate={win_rate*100:5.1f}%")

print("
=== SIGNAL PERFORMANCE ===")
calc_metrics(mom_returns.dropna(), 'Momentum')
calc_metrics(mr_returns.dropna(), 'Mean Reversion')
calc_metrics(rsi_returns.dropna(), 'RSI')
calc_metrics(returns.dropna(), 'Buy & Hold')

## 6. Signal Blending

In [ ]:
# Blend multiple signals
blender = SignalBlender(
    signals=[
        MomentumSignal(lookback=60, skip=21),
        MeanReversionSignal(lookback=20),
        RSISignal(lookback=14),
    ],
    weights=[0.4, 0.3, 0.3]  # 40% momentum, 30% mean reversion, 30% RSI
)

blended_signal = blender.compute(prices)
blended_positions = blender.signals[0].to_positions(blended_signal)

# Compare blended vs individual
blended_returns = blended_positions.shift(1) * returns

plt.figure(figsize=(12, 6))
plt.plot((1 + mom_returns.dropna()).cumprod(), label='Momentum', alpha=0.7)
plt.plot((1 + mr_returns.dropna()).cumprod(), label='Mean Reversion', alpha=0.7)
plt.plot((1 + blended_returns.dropna()).cumprod(), label='Blended', linewidth=2)
plt.plot((1 + returns.dropna()).cumprod(), label='Buy & Hold', alpha=0.5)
plt.legend()
plt.title('Blended Signal vs Individual Signals')
plt.grid(True, alpha=0.3)
plt.show()

print("
=== BLENDED PERFORMANCE ===")
calc_metrics(blended_returns.dropna(), 'Blended Signal')

## 7. Multi-Asset Signal Research

In [ ]:
# Load multiple assets
print("Loading multiple assets...")
asset_data = {}
for name, ticker in assets.items():
    data = yf.download(ticker, start=start_date, end=end_date, progress=False)
    if len(data) > 0:
        asset_data[name] = data['Close']

# Combine into DataFrame
prices_multi = pd.DataFrame(asset_data)
prices_multi = prices_multi.dropna()

print(f"Loaded {len(prices_multi)} days of data for {len(prices_multi.columns)} assets")
print(prices_multi.columns.tolist())

# Compute momentum for each asset
momentum = MomentumSignal(lookback=60)
mom_multi = momentum.compute(prices_multi)

# Show momentum signals
plt.figure(figsize=(12, 6))
for col in mom_multi.columns:
    plt.plot(mom_multi[col].dropna().tail(500), label=col, alpha=0.7)
plt.legend()
plt.title('Momentum Signal by Asset')
plt.grid(True, alpha=0.3)
plt.show()

## 8. Summary

In [ ]:
print("""
=== KEY TAKEAWAYS ===

1. **Signal Types**
   - Momentum: trend-following (buy when going up)
   - Mean Reversion: contrarian (buy when oversold)
   - RSI/MACD: oscillators for timing entries
   - Bollinger: volatility-based entries

2. **Signal to Positions**
   - Each signal has to_positions() method
   - Returns -1 (short) to 1 (long)
   - Can customize position sizing

3. **Signal Blending**
   - Combine multiple signals
   - Weight by conviction
   - Z-score normalization for fair blending

4. **Multi-Asset**
   - Signals work on multi-asset DataFrames
   - Compute once, apply to all assets

5. **Next Steps**
   - Use Walk-Forward analysis to validate signals
   - Add more sophisticated signals (fundamentals)
   - Build portfolio with signal combination
""")